# Training — Zoobot ConvNeXt (Bloque A · Domain Shift Mínimo)

Notebook reutilizable que ejecuta únicamente llamadas a funciones definidas en `src/`.
Encoder **ConvNeXt-nano** preentrenado sobre ~10 M galaxias de Galaxy Zoo (Zoobot 2.0).

> **Referencia:** Walmsley et al. 2023 — *Scaling Laws for Galaxy Images* — arXiv:2404.02973  
> **Pesos:** `hf_hub:mwalmsley/zoobot-encoder-convnext_nano` (HuggingFace Hub)

## 0) Dependencias específicas

Este modelo requiere `zoobot` (que instala `timm` y `pytorch-lightning`) además de las dependencias base del proyecto.

In [ ]:
# Instalar Zoobot si no está disponible (ejecutar solo la primera vez)
# !poetry add "zoobot[pytorch]"

try:
    import timm
    print(f"timm {timm.__version__} — OK")
except ImportError:
    raise ImportError("Ejecuta: poetry add 'zoobot[pytorch]'")


## 1) Imports y configuración básica

In [ ]:
import sys
from pathlib import Path
import torch
import torch.nn as nn
import torch.optim as optim

# Asegurar que la raíz del repo está en sys.path (notebook ubicado en `baseline/`)
sys.path.append(str(Path("..").resolve()))

from src import (
    CONFIG,
    CATALOG_PATH,
    FITS_DIR,
    build_dataloaders,
    calculate_class_weights,
    compute_classification_metrics,
    create_model_dir,
    evaluate_and_save,
    generate_gradcam_visualization,
    load_catalog,
    save_learning_curves,
    save_training_checkpoint,
    save_training_metadata,
    set_global_seed,
    split_catalog,
    train_loop,
    get_device,
)

# Builder específico para este experimento
from src.galaxy_model_builders import build_zoobot_convnext

print("Imports OK")


## 2) Semilla y dispositivo

In [ ]:
set_global_seed(CONFIG["seed"])
device = get_device()
print(f"Using device: {device}")


## 3) Carga del catálogo y división estratificada

In [ ]:
from pathlib import Path

catalog_path = Path(CATALOG_PATH)
report_path = Path("..") / "data" / "processed" / "dataset_validation_report.csv"

df = load_catalog(catalog_path, report_path=report_path)
train_df, val_df, test_df = split_catalog(df, seed=CONFIG["seed"])

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


## 4) Datasets y DataLoaders

> **Nota:** Zoobot ConvNeXt acepta cualquier tamaño de entrada.
> El `CONFIG["image_size"]` del proyecto se usa sin modificación.

In [ ]:
train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = build_dataloaders(
    train_df,
    val_df,
    test_df,
    FITS_DIR,
    CONFIG,
)

print("Datasets and DataLoaders ready")


## 5) Construcción del modelo (Zoobot ConvNeXt-nano)

| Parámetro | Valor | Descripción |
|---|---|---|
| `variant` | `convnext_nano` | Encoder 256-d, 4.5 M params totales |
| `freeze_backbone` | `True` | Solo entrena la cabeza clasificadora |
| `head_hidden` | 128 | Igual que ResNet-18 baseline |
| `head_dropout` | 0.3 | Igual que ResNet-18 baseline |

Para probar variantes más grandes cambia `variant` a `convnext_tiny`, `convnext_small`, etc.

In [ ]:
# Variante y nombre del experimento
ZOOBOT_VARIANT = "convnext_nano"   # opciones: convnext_pico | nano | tiny | small | base | large
model_name = f"zoobot_{ZOOBOT_VARIANT}"

model, trainable_params, frozen_params = build_zoobot_convnext(
    num_classes=CONFIG["num_classes"],
    variant=ZOOBOT_VARIANT,
    freeze_backbone=CONFIG.get("freeze_backbone", True),
    head_hidden=128,
    head_dropout=0.3,
    device=device,
)

print(f"Modelo      : Zoobot ConvNeXt-{ZOOBOT_VARIANT}")
print(f"Preentren.  : ~10M galaxias Galaxy Zoo (DESI / DECaLS / Rings)")
print(f"Encoder dim : 256-d")
print(f"Trainable params: {trainable_params:,} | Frozen params: {frozen_params:,}")


## 6) Pérdida y optimizador

Idéntico al baseline. El `filter(lambda p: p.requires_grad, ...)` garantiza que el optimizer **solo actualice la cabeza** mientras el backbone Zoobot permanece congelado.

In [ ]:
class_weights = calculate_class_weights(train_df, CONFIG["num_classes"], device)
criterion = torch.nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=CONFIG["learning_rate"],
    weight_decay=CONFIG["weight_decay"],
)

print("Criterion and optimizer ready")


## 7) Entrenamiento y guardado de artefactos

In [ ]:
model_dir = create_model_dir(model_name)
checkpoint_path = model_dir / "checkpoint.pth"
history_path = model_dir / "history.json"

if checkpoint_path.exists() and history_path.exists():
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    history = checkpoint.get("history", {})
    print(f"Loaded existing checkpoint from {checkpoint_path}")
else:
    model, history = train_loop(
        model, train_loader, val_loader, criterion, optimizer, device,
        epochs=CONFIG["epochs"]
    )
    save_training_checkpoint(
        model,
        optimizer,
        history,
        CONFIG,
        class_weights,
        trainable_params,
        frozen_params,
        model_dir,
    )

training_meta = {
    "model_name": model_name,
    "architecture": "Zoobot ConvNeXt",
    "variant": ZOOBOT_VARIANT,
    "pretrained_on": "~10M galaxies Galaxy Zoo (DESI / DECaLS / Rings)",
    "domain_shift": "minimal",
    "encoder_dim": 256,
    "epochs": CONFIG["epochs"],
    "config": CONFIG,
    "trainable_params": trainable_params,
    "frozen_params": frozen_params,
    "dataset_split": {"train": len(train_df), "val": len(val_df), "test": len(test_df)},
    "catalog_path": str(catalog_path),
    "fits_dir": str(FITS_DIR),
    "checkpoint_file": checkpoint_path.name,
    "history_file": history_path.name,
    "reference": "Walmsley et al. 2023, arXiv:2404.02973",
    "weights_hub": f"hf_hub:mwalmsley/zoobot-encoder-{ZOOBOT_VARIANT}",
}

save_training_metadata(model_dir, history, training_meta)
print(f"Model artifacts saved under: {model_dir}")


## 8) Evaluación final en Test Set

In [ ]:
eval_dir = model_dir / "evaluation"
sample_names = test_df["name"].tolist()
metrics = evaluate_and_save(
    model,
    test_loader,
    device,
    compute_classification_metrics,
    eval_dir,
    sample_names=sample_names,
)

print("Test metrics:")
for key, value in metrics.items():
    print(f"{key}: {value}")


## 9) Curvas de aprendizaje

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg


In [ ]:
learning_curves_path = save_learning_curves(history, model_dir)
print(f"Learning curves saved: {learning_curves_path}")

img = mpimg.imread(learning_curves_path)
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis("off")
plt.title(f"Learning curves — Zoobot {ZOOBOT_VARIANT}")
plt.show()


## 10) Grad-CAM de ejemplo

> **Nota Zoobot:** El encoder ConvNeXt es compatible con Grad-CAM sobre sus capas convolucionales.
> Si `generate_gradcam_visualization` usa `model.layer4` (ResNet), adapta el `target_layer`
> a `model.encoder.stages[-1]` para ConvNeXt.

In [ ]:
gradcam_dir = model_dir / "gradcam"
gradcam_dir.mkdir(parents=True, exist_ok=True)
gradcam_path = gradcam_dir / "gradcam_positive_label1.png"

saved_path = generate_gradcam_visualization(
    model, test_dataset, device, target_label=1, save_path=gradcam_path
)
print(f"Grad-CAM saved: {saved_path}")

img = mpimg.imread(saved_path)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis("off")
plt.title("Grad-CAM for label 1 — Zoobot ConvNeXt")
plt.show()


## 11) Notas

- Este notebook no define funciones nuevas: usa utilidades reutilizables de `src/` más `galaxy_model_builders.py`.
- Para cambiar de variante ConvNeXt edita únicamente `ZOOBOT_VARIANT` en la celda 5.
- Para fine-tuning completo del backbone cambia `freeze_backbone=False` y reduce el `learning_rate` a `1e-5` o `5e-6`.
- Comparación directa con baseline: los artefactos se guardan en `model_dir` con el mismo esquema de directorios que `resnet18_baseline`.
